# ETL — VISp MET-types Taxonomy (cluster reference)

Registers the **VISp MET-types taxonomy** as a global cluster reference. Writes `algorithmrun/`, `clusterhierarchy/`, `cluster/`, `hierarchycategory/`. **Out of scope:** no `DataItem` registration.

Source: `met_type_colors.json` (45 MET-type labels, leaf-only colors). Two real levels (class → cluster) with a synthetic `cell` root. Class-level colors sourced from Tasic's `anno.feather` for visual consistency. Schema caveats already documented in `etl_tasic_01_cluster.ipynb`; not repeated here.

In [1]:
from pathlib import Path
import json

import pandas as pd
import polars as pl

from connects_common_connectivity.models import (
    AlgorithmRun,
    Cluster,
    ClusterHierarchy,
    HierarchyCategory,
)
from connects_common_connectivity.config import output_root
from connects_common_connectivity.io import write_models


In [2]:
INPUT_JSON    = "/data/visp-patchseq-taxonomy-info/met_type_colors.json"
TASIC_FEATHER = "/data/visp-patchseq-taxonomy-info/anno.feather"
OUTPUT_ROOT   = output_root()
HIERARCHY_ID  = "visp_met_types_taxonomy"
RUN_ID        = "visp_met_types_clustering"
ROOT_ID       = "cell"

assert Path(INPUT_JSON).exists(),    f"Input not found: {INPUT_JSON}"
assert Path(TASIC_FEATHER).exists(), f"Input not found: {TASIC_FEATHER}"
print(f"INPUT_JSON    : {INPUT_JSON}")
print(f"TASIC_FEATHER : {TASIC_FEATHER}")
print(f"OUTPUT_ROOT   : {OUTPUT_ROOT}")
print(f"HIERARCHY_ID  : {HIERARCHY_ID}")
print(f"RUN_ID        : {RUN_ID}")

INPUT_JSON    : /data/visp-patchseq-taxonomy-info/met_type_colors.json
TASIC_FEATHER : /data/visp-patchseq-taxonomy-info/anno.feather
OUTPUT_ROOT   : ../scratch/em_patchseq_wnm_v2/
HIERARCHY_ID  : visp_met_types_taxonomy
RUN_ID        : visp_met_types_clustering


## Load + parse

In [3]:
with open(INPUT_JSON) as f:
    met_colors: dict[str, str] = json.load(f)

# Class-color reuse: pull GABAergic / Glutamatergic colors from Tasic, matching the
# reference notebook (process_patchseq_taxonomy_info.ipynb).
tasic_df = pd.read_feather(TASIC_FEATHER)
tasic_class_colors = dict(zip(tasic_df.class_label, tasic_df.class_color))
GABA_COLOR = tasic_class_colors["GABAergic"]
GLUT_COLOR = tasic_class_colors["Glutamatergic"]

# Leaf split: "MET" in label → GABAergic, otherwise → Glutamatergic.
gaba_met_types = [t for t in met_colors if "MET" in t]
glut_met_types = [t for t in met_colors if "MET" not in t]

assert len(gaba_met_types) + len(glut_met_types) == len(met_colors), "leaves don't partition cleanly"
assert set(gaba_met_types).isdisjoint(glut_met_types)
assert len(gaba_met_types) == 28
assert len(glut_met_types) == 17

print(f"met_types: total={len(met_colors)}  gaba={len(gaba_met_types)}  glut={len(glut_met_types)}")
print(f"class colors (from Tasic): GABAergic={GABA_COLOR}  Glutamatergic={GLUT_COLOR}")

met_types: total=45  gaba=28  glut=17
class colors (from Tasic): GABAergic=#EF4136  Glutamatergic=#27AAE1


## `HierarchyCategory` — 3 rows (`major_class`/`class`/`cluster`); no `subclass` for this taxonomy

In [4]:
# Stored as str (top-level `level` slot has no integer range; see Tasic notebook caveats).
category_rows = [
    HierarchyCategory(id="cluster",     description="Leaf cluster (cell type / MET-type).",  level="0"),
    HierarchyCategory(id="class",       description="Top-level cell class.",                 level="2"),
    HierarchyCategory(id="major_class", description="Synthetic root grouping all classes.",  level="3"),
]
CATEGORY_IDS = [c.id for c in category_rows]

result = write_models(category_rows, output_root=OUTPUT_ROOT)
print(f"HierarchyCategory written: {result.rows_written} rows")


HierarchyCategory written: 3 rows


In [5]:
# Cross-taxonomy check: confirm Tasic's `subclass` row is undisturbed (it's not in this notebook's predicate).
all_categories = pl.read_delta(OUTPUT_ROOT + "hierarchycategory/").sort("id")
print(all_categories)
assert set(all_categories["id"].to_list()) >= {"cluster","class","major_class","subclass"}, (
    "expected `subclass` row from Tasic notebook to remain"
)

shape: (4, 3)
┌─────────────┬─────────────────────────────────┬───────┐
│ id          ┆ description                     ┆ level │
│ ---         ┆ ---                             ┆ ---   │
│ str         ┆ str                             ┆ str   │
╞═════════════╪═════════════════════════════════╪═══════╡
│ class       ┆ Top-level cell class.           ┆ 2     │
│ cluster     ┆ Leaf cluster (cell type / MET-… ┆ 0     │
│ major_class ┆ Synthetic root grouping all cl… ┆ 3     │
│ subclass    ┆ Subclass of cell types.         ┆ 1     │
└─────────────┴─────────────────────────────────┴───────┘


## `AlgorithmRun` — 1 row

In [6]:
run_row = AlgorithmRun(
    id=RUN_ID,
    algorithm_name="VISp MET-types taxonomy (Patch-seq morpho-electric-transcriptomic types)",
    algorithm_version="2021",
    score_description=None,
    distance_description=None,
    # input_dataset omitted: MET-type cells are not registered in this codebase.
    # produced_hierarchies omitted: schema declares it as inlined dict[id, ClusterHierarchy].
)

result = write_models([run_row], output_root=OUTPUT_ROOT)
print(f"AlgorithmRun written: {result.rows_written} rows")


AlgorithmRun written: 1 rows


In [7]:
verify_run = pl.read_delta(OUTPUT_ROOT + "algorithmrun/").filter(pl.col("id") == RUN_ID)
print(verify_run.shape)
assert verify_run.shape[0] == 1
print(verify_run)

(1, 9)
shape: (1, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ algorithm ┆ algorithm ┆ json_obje ┆ … ┆ input_dat ┆ produced_ ┆ score_des ┆ distance │
│ ---       ┆ _name     ┆ _version  ┆ ct        ┆   ┆ aset      ┆ hierarchi ┆ cription  ┆ _descrip │
│ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ es        ┆ ---       ┆ tion     │
│           ┆ str       ┆ str       ┆ str       ┆   ┆ str       ┆ ---       ┆ str       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆ str       ┆           ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ visp_met_ ┆ VISp      ┆ 2021      ┆ null      ┆ … ┆ null      ┆ null      ┆ null      ┆ null     │
│ types_clu ┆ MET-types ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ stering   ┆ taxonomy  ┆           ┆           ┆   ┆           ┆     

## `Cluster` — 48 rows (1 synthetic root + 2 classes + 45 leaves)

In [8]:
cluster_rows: list[Cluster] = []

# Synthetic root: depth 0.
cluster_rows.append(Cluster(
    id=ROOT_ID,
    hierarchy_id=HIERARCHY_ID,
    parent=None,
    children=["GABAergic", "Glutamatergic"],
    level=0,
    hex_color="#000000",
    hierarchy_category="major_class",
))

# Class level: depth 1.
cluster_rows.append(Cluster(
    id="GABAergic",
    hierarchy_id=HIERARCHY_ID,
    parent=ROOT_ID,
    children=gaba_met_types,
    level=1,
    hex_color=GABA_COLOR,
    hierarchy_category="class",
))
cluster_rows.append(Cluster(
    id="Glutamatergic",
    hierarchy_id=HIERARCHY_ID,
    parent=ROOT_ID,
    children=glut_met_types,
    level=1,
    hex_color=GLUT_COLOR,
    hierarchy_category="class",
))

# Leaf level: depth 2.
for t in gaba_met_types:
    cluster_rows.append(Cluster(
        id=t, hierarchy_id=HIERARCHY_ID, parent="GABAergic", children=[],
        level=2, hex_color=met_colors[t], hierarchy_category="cluster",
    ))
for t in glut_met_types:
    cluster_rows.append(Cluster(
        id=t, hierarchy_id=HIERARCHY_ID, parent="Glutamatergic", children=[],
        level=2, hex_color=met_colors[t], hierarchy_category="cluster",
    ))

assert len(cluster_rows) == 1 + 2 + 45 == 48
print(f"Cluster rows built: {len(cluster_rows)}")
result = write_models(cluster_rows, output_root=OUTPUT_ROOT)
print(f"Cluster written: {result.rows_written} rows")

Cluster rows built: 48


Cluster written: 48 rows


In [9]:
verify_clu = pl.read_delta(OUTPUT_ROOT + "cluster/").filter(pl.col("hierarchy_id") == HIERARCHY_ID)
print(verify_clu.shape)
assert verify_clu.shape[0] == 48
assert verify_clu.filter(pl.col("id") == ROOT_ID).shape[0] == 1
print(verify_clu.group_by("hierarchy_category").len().sort("hierarchy_category"))

(48, 9)
shape: (3, 2)
┌────────────────────┬─────┐
│ hierarchy_category ┆ len │
│ ---                ┆ --- │
│ str                ┆ u32 │
╞════════════════════╪═════╡
│ class              ┆ 2   │
│ cluster            ┆ 45  │
│ major_class        ┆ 1   │
└────────────────────┴─────┘


## `ClusterHierarchy` — 1 row

In [10]:
hierarchy_row = ClusterHierarchy(
    id=HIERARCHY_ID,
    run=RUN_ID,
    root=ROOT_ID,
    clusters=[c.id for c in cluster_rows],
)
result = write_models([hierarchy_row], output_root=OUTPUT_ROOT)
print(f"ClusterHierarchy written: {result.rows_written} rows")

ClusterHierarchy written: 1 rows


In [11]:
verify_h = pl.read_delta(OUTPUT_ROOT + "clusterhierarchy/").filter(pl.col("id") == HIERARCHY_ID)
print(verify_h.shape)
assert verify_h.shape[0] == 1
row = verify_h.row(0, named=True)
assert row["root"] == ROOT_ID and row["run"] == RUN_ID
assert len(row["clusters"]) == 48
print(f"root={row['root']}  run={row['run']}  clusters={len(row['clusters'])}")

(1, 4)
root=cell  run=visp_met_types_clustering  clusters=48


## Summary

Written under `../scratch/em_patchseq_wnm_v1/`:

| Table | Rows | Predicate |
|---|---|---|
| `algorithmrun/` | +1 (`visp_met_types_clustering`) | `id = RUN_ID` |
| `clusterhierarchy/` | +1 (`visp_met_types_taxonomy`) | `id = HIERARCHY_ID` |
| `cluster/` | +48 | `hierarchy_id = HIERARCHY_ID` (partitioned) |
| `hierarchycategory/` | 3 (overwrite-merged with Tasic's 4) | `id IN (...)` |

Coexists alongside the Tasic taxonomy in the same global tables. Idempotent.